
# Arm scan: positioner 회전 중심과 arm 길이

beta 를 고정하고 alpha 를 한 바퀴 돌리며 찍은 이미지들로, positioner 마다 fiber 끝의 궤적에 원을 맞춘다 (`analysis/armcal.py`).
원의 중심 = 회전 중심, 반지름 R = |a₁ + a₂e^{iφ}|. beta = 0 (R = a₁ + a₂) 와 beta = 180 (R = a₂ − a₁) 두 번 돌리면 `fit_arms` 로 a₁, a₂ 를 따로 구한다.
beta 만 돌리면 팔꿈치 중심 반지름 a₂ 의 원.

이미지마다 fiducial 이 같이 찍혀 있으면 이미지마다 변환을 새로 맞춘다 (카메라가 흘러가도 됨). 결과는 `Measured_Geometry` 형식으로 써서 패키지의 실측 회전 중심으로 쓸 수 있다.


## 촬영 (take_exposure.py)
노출은 `kspec_metrology/notebooks/take_exposure.py` 로 직접 찍는다 (노출시간 [ms]; 이름 끝에 `_0` 이 자동으로 붙는다; `--dry-run` 은 이름만 확인).
한 폴더에 alpha 단계마다 fiber 노출 한 장씩 찍는다 (이름에 각도를 넣으면 정렬이 된다). fiducial 은 같은 폴더에 전용 노출 한 벌을 처음(과 끝)에 찍어 둔다:
```bash
python kspec_metrology/notebooks/take_exposure.py --fiber 400 --fid 2 10 --out /data/scan_b0/ref      # fiducial 기준 (ref_fid2_10ms_0.fits)
for a in 000 030 060 090 120 150 180 210 240 270 300 330; do
    # positioner alpha 를 $a 도로 보낸 뒤
    python kspec_metrology/notebooks/take_exposure.py --exptime 400 --out /data/scan_b0/alpha_$a.fits
done
```
→ `SCAN_IMAGES = sorted(glob.glob('/data/scan_b0/alpha_*_0.fits'))`, `SCAN_FID = ['/data/scan_b0/ref_fid2_10ms_0.fits']`.
fiducial 이 fiber 노출에도 같이 찍혀 있으면 `SCAN_FID = []` 로 두어 이미지마다 변환을 새로 맞출 수 있다 (포화된 fiducial 은 위치가 수십 µm 틀어질 수 있다).
beta = 180 으로 한 번 더 돌리면 `/data/scan_b180/` 에 같은 식으로.


In [ ]:

import os, sys, json, glob
PKG_DIR  = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))   # 이 노트북이 kspec_metrology/notebooks/ 에 있을 때 저장소 루트
OUT_DIR  = os.path.expanduser('~/metrology_out')                   # <- 결과를 쓸 폴더

# ---- 설정: 바꾸면 커널을 재시작한다 (표와 제외 목록은 import 때 읽힌다) ----
FIBER_TABLE   = None            # None = 패키지 기본 표 (config73). 다른 표면 경로
# 빈 리스트 [] 는 패키지 기본값(analysis/fiber_config.py, 설치표 2026-10-12 기준) 을 그대로 쓴다. 적으면 그것이 우선.
BROKEN_FIBERS = []              # 빛이 안 나는 positioner 홀 ID. 기본: 분광기에 fiber 가 안 연결된 33 홀, 끊어진 G4 포함 (= cfg.FIBER_EXCLUDE)
FIXED_FIBERS  = []              # 켜져 있지만 움직이지 않는 positioner 홀 ID. 기본: ['G7', 'H8', 'J6'] (sky fiber 가 꽂힌 고정 positioner)
SKY_FIBERS    = []              # 표에서 -9 인 홀의 sky fiber. 기본: ['B7', 'H1', 'N7']
ANGLE_SIGN    = (1, 1)          # 출력 각도 = 부호 x 기하각 (alpha 반시계 양, beta >= 0)
ANGLE_ZERO    = (30, 150)       # positioner 명령각 영점: 기하각 (30, 150) = 명령 (0, 0). 출력·각도 입력 모두 명령각 = 부호 x 기하각 - 영점

if FIBER_TABLE:
    os.environ['KSPEC_FIBER_TABLE'] = FIBER_TABLE
os.environ['KSPEC_FIBER_EXCLUDE'] = ','.join(BROKEN_FIBERS)
os.environ['KSPEC_FIBER_FIXED']   = ','.join(FIXED_FIBERS)
os.environ['KSPEC_SKY_FIBERS']    = ','.join(SKY_FIBERS)
os.environ['KSPEC_ANGLE_SIGN']    = f'{ANGLE_SIGN[0]},{ANGLE_SIGN[1]}'
os.environ['KSPEC_ANGLE_ZERO']    = f'{ANGLE_ZERO[0]},{ANGLE_ZERO[1]}'
os.environ['KSPEC_USE_MEASURED']  = '1'
os.makedirs(OUT_DIR, exist_ok=True)
sys.path.insert(0, PKG_DIR)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 80
from kspec_metrology import lab
from kspec_metrology.analysis import fiber_config as cfg
print(f'fiber table {cfg.FIBER_TABLE_PATH}: {len(cfg.fiber_ids())} positioners')
print('broken', cfg.FIBER_EXCLUDE, '| fixed', cfg.FIBER_FIXED, '| sky', cfg.SKY_FIBERS, '| angle sign', cfg.ANGLE_SIGN, '| angle zero', cfg.ANGLE_ZERO)


In [ ]:

MAGNIFICATION  = None           # camera -> focal 배율 초기값. None = 패키지 기본 (2026-01 commissioning fit -8.86, 설치된 실카메라). 10/17 시험 셋업 영상은 8.40
MODEL          = 'full+affine'  # 본 장비. fiducial 30개 기준 leave-one-out 이 가장 좋았다 (README 표). 실험실 광학계는 'radial+affine'
NWINDOW        = 40
NITER_RECENTER = 1
THRESHOLD      = 1e3            # 시작 문턱 [ADU, 배경 포함]; 검출 개수에 맞춰 자동으로 올리고 내린다
MIN_DIST       = 80             # 이보다 가까운 검출은 하나로 합침 [px]
FINDER_OPTS    = {'minarea': 9, 'deblend_cont': 1.0}
N_EXTRA        = 0              # FIXED/SKY 외의 여분 밝은 점 (반사광 등)
SAT_LEVEL      = 65000.
TILE           = 'tile01'


In [ ]:

from kspec_metrology.analysis import armcal
SCAN_IMAGES = sorted(glob.glob('/path/to/scan/*_0.fits'))      # alpha 를 돌리며 찍은 이미지들 (같은 폴더, ...0.fits)
SCAN_FID    = []                                             # fiducial 전용 노출 (없으면 각 이미지의 fiducial)
print(len(SCAN_IMAGES), 'images')
res_scan = armcal.arm_calibration(SCAN_IMAGES, fid_images=SCAN_FID, magnification=MAGNIFICATION, model=MODEL,
                                  n_extra=N_EXTRA, threshold=THRESHOLD, min_dist=MIN_DIST, finder_opts=FINDER_OPTS)
T = res_scan.table
print(f"fitted {int(np.isfinite(T.R).sum())}/{len(T)}; R median {T.R.median():.3f} mm; circle rms median {T.rms.median()*1e3:.0f} um; "
      f"centre - design: median {np.hypot(T.dx, T.dy).median()*1e3:.0f} um")
T.round(4)


In [ ]:

armcal.plot_arm_calibration(res_scan);



## arm 길이 (두 beta 에서 돌렸을 때) 와 실측 기하 파일


In [ ]:

# res_b0  = armcal.arm_calibration(IMAGES_BETA0,   ...);  res_b180 = armcal.arm_calibration(IMAGES_BETA180, ...)
# a1, a2 = armcal.fit_arms(res_b0.table.R, res_b180.table.R)
# arms = {i: (a, b) for i, a, b in zip(res_b0.table.index, a1, a2)}
arms = None
out = armcal.write_measured_geometry(res_scan, os.path.join(OUT_DIR, 'Measured_Geometry_scan.txt'),
                                     template=os.path.join(PKG_DIR, 'kspec_metrology', 'analysis', 'Measured_Geometry.txt'), arms=arms)
print(out); print(open(out).read()[:800])
